In [1]:
import json
from pathlib import Path
import pandas as pd

# Locate the project directory
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "knowledge_base").exists():
    PROJECT_DIR = CURRENT_DIR
else:
    PROJECT_DIR = CURRENT_DIR.parent

KNOWLEDGE_BASE_DIR = PROJECT_DIR / "knowledge_base"

RULES_PATH = KNOWLEDGE_BASE_DIR / "conduct_rules.json"
PROCEDURE_PATH = KNOWLEDGE_BASE_DIR / "procedure_rules.json"

print("Project directory:", PROJECT_DIR)
print("Knowledge base directory:", KNOWLEDGE_BASE_DIR)

Project directory: c:\Users\user\Desktop\ConductIQ
Knowledge base directory: c:\Users\user\Desktop\ConductIQ\knowledge_base


In [2]:
# Load conduct rules
with open(RULES_PATH, "r", encoding="utf-8") as file:
    conduct_data = json.load(file)

# Load procedures and deadlines
with open(PROCEDURE_PATH, "r", encoding="utf-8") as file:
    procedure_data = json.load(file)

conduct_rules = conduct_data["rules"]
procedures = procedure_data["procedures"]
deadlines = procedure_data.get("deadlines", [])

print("Knowledge bases loaded successfully.")
print("Conduct rules:", len(conduct_rules))
print("Procedures:", len(procedures))
print("Deadlines:", len(deadlines))

Knowledge bases loaded successfully.
Conduct rules: 29
Procedures: 14
Deadlines: 4


In [3]:
EVIDENCE_REQUIREMENTS = {
    "conduct_rule": [
        "rule_id",
        "rule_name",
        "description",
        "section_number",
        "source_page"
    ],
    "procedure": [
        "procedure_id",
        "procedure_name",
        "description",
        "section_number",
        "source_page"
    ],
    "deadline": [
        "deadline_id",
        "action",
        "deadline_value",
        "deadline_unit",
        "trigger",
        "section_number",
        "source_page"
    ]
}

print("Evidence requirements defined.")

Evidence requirements defined.


In [4]:
# Implement the edidence validator
class EvidenceValidator:

    def __init__(self, requirements=None):
        self.requirements = requirements or EVIDENCE_REQUIREMENTS

    @staticmethod
    def is_valid_page(page):
        try:
            return int(page) > 0
        except (TypeError, ValueError):
            return False

    def validate_record(self, record, evidence_type):

        if evidence_type not in self.requirements:
            return {
                "valid": False,
                "issues": [f"Unknown evidence type: {evidence_type}"]
            }

        if not isinstance(record, dict):
            return {
                "valid": False,
                "issues": ["Evidence record must be a dictionary."]
            }

        issues = []

        # Check required fields
        for field in self.requirements[evidence_type]:

            value = record.get(field)

            if value is None or str(value).strip() == "":
                issues.append(f"Missing required field: {field}")

        # Check printed page number
        if not self.is_valid_page(record.get("source_page")):
            issues.append("Invalid or missing printed page number.")

        # Check deadline value
        if evidence_type == "deadline":

            try:
                if int(record.get("deadline_value")) <= 0:
                    issues.append("Deadline value must be positive.")
            except (TypeError, ValueError):
                issues.append("Deadline value must be a valid number.")

        return {
            "valid": len(issues) == 0,
            "issues": issues
        }

    def validate_collection(self, records, evidence_type):

        results = []

        for record in records:

            validation = self.validate_record(
                record,
                evidence_type
            )

            results.append({
                "evidence_id": (
                    record.get("rule_id")
                    or record.get("procedure_id")
                    or record.get("deadline_id")
                ),
                "evidence_type": evidence_type,
                "valid": validation["valid"],
                "issues": validation["issues"]
            })

        return results


validator = EvidenceValidator()

print("Evidence Validator initialized successfully.")

Evidence Validator initialized successfully.


In [5]:
# Validate all knowledge-base records
# Validate conduct rules
rule_validation = validator.validate_collection(
    conduct_rules,
    "conduct_rule"
)

# Validate procedures
procedure_validation = validator.validate_collection(
    procedures,
    "procedure"
)

# Validate deadlines
deadline_validation = validator.validate_collection(
    deadlines,
    "deadline"
)

# Combine validation results
all_validation_results = (
    rule_validation
    + procedure_validation
    + deadline_validation
)

validation_df = pd.DataFrame(all_validation_results)

print("Evidence validation completed.")

display(
    validation_df.groupby(
        ["evidence_type", "valid"]
    ).size().unstack(fill_value=0)
)

Evidence validation completed.


valid,True
evidence_type,
conduct_rule,29
deadline,4
procedure,14


In [6]:
invalid_records = validation_df[
    validation_df["valid"] == False
]

if invalid_records.empty:
    print("All evidence records passed structural validation.")
else:
    print("Records requiring correction:")
    display(invalid_records)

All evidence records passed structural validation.


In [7]:
def validate_citation(record):

    section = record.get("section_number")
    page = record.get("source_page")

    issues = []

    if not section or not str(section).strip():
        issues.append("Missing section number.")

    if not EvidenceValidator.is_valid_page(page):
        issues.append("Missing or invalid printed page number.")

    return {
        "valid": len(issues) == 0,
        "section_number": section,
        "source_page": page,
        "issues": issues
    }


def format_citation(record):

    citation = validate_citation(record)

    if not citation["valid"]:
        return "Citation unavailable"

    return (
        f"Section {citation['section_number']}, "
        f"printed page {citation['source_page']}"
    )


# Test citation formatting
for record in procedures[:3]:

    print(
        record["procedure_name"],
        "->",
        format_citation(record)
    )

Purpose and principles of disciplinary procedures -> Section Chapter V, printed page 19
College Disciplinary Committee -> Section 5.1, printed page 19
CDC functions -> Section 5.1.2, printed page 19


In [8]:
# Create the vidence formatter 
def format_evidence(record, evidence_type):

    validation = validator.validate_record(
        record,
        evidence_type
    )

    if not validation["valid"]:
        return None

    if evidence_type == "conduct_rule":

        evidence_id = record["rule_id"]
        title = record["rule_name"]
        evidence_text = record["description"]

    elif evidence_type == "procedure":

        evidence_id = record["procedure_id"]
        title = record["procedure_name"]
        evidence_text = record["description"]

    elif evidence_type == "deadline":

        evidence_id = record["deadline_id"]
        title = record["action"]

        evidence_text = (
            f"{record['action']} within "
            f"{record['deadline_value']} "
            f"{record['deadline_unit']}. "
            f"Trigger: {record['trigger']}."
        )

        if record.get("qualification"):
            evidence_text += (
                " Qualification: " + record["qualification"]
            )

    else:
        return None

    return {
        "evidence_id": evidence_id,
        "evidence_type": evidence_type,
        "title": title,
        "evidence_text": evidence_text,
        "section_number": record["section_number"],
        "source_page": int(record["source_page"]),
        "citation": format_citation(record),
        "retrieval_score": record.get("retrieval_score"),
        "query_coverage": record.get("query_coverage"),
        "source_quote": record.get("source_quote")
    }



print("Evidence formatter created successfully.")

Evidence formatter created successfully.


In [9]:
# Implement the evidence assessment and abstention handler
def assess_evidence(evidence_records):

    if not evidence_records:
        return {
            "status": "insufficient_evidence",
            "can_answer": False,
            "message": (
                "I could not find sufficient supporting information "
                "in the available knowledge base. Please contact the "
                "appropriate College authority or Student Services Officer."
            )
        }

    return {
        "status": "evidence_found",
        "can_answer": True,
        "message": "Supporting evidence was retrieved."
    }


print("Evidence assessment function created successfully.")

Evidence assessment function created successfully.


In [11]:
# Build the final evidence packet

def build_evidence_packet(
    rule_results=None,
    procedure_results=None,
    deadline_results=None
):

    rule_results = rule_results or []
    procedure_results = procedure_results or []
    deadline_results = deadline_results or []

    evidence_packet = []

    # Format conduct-rule evidence
    for record in rule_results:

        evidence = format_evidence(
            record,
            "conduct_rule"
        )

        if evidence:
            evidence_packet.append(evidence)

    # Format procedure evidence
    for record in procedure_results:

        evidence = format_evidence(
            record,
            "procedure"
        )

        if evidence:
            evidence_packet.append(evidence)

    # Format deadline evidence
    for record in deadline_results:

        evidence = format_evidence(
            record,
            "deadline"
        )

        if evidence:
            evidence_packet.append(evidence)

    assessment = assess_evidence(evidence_packet)

    return {
        "status": assessment["status"],
        "can_answer": assessment["can_answer"],
        "message": assessment["message"],
        "evidence_count": len(evidence_packet),
        "evidence": evidence_packet
    }


print("Evidence packet builder created successfully.")

Evidence packet builder created successfully.


In [12]:
# Test with actual knowledge-base records

# Select representative records
sample_rule = next(
    (r for r in conduct_rules if r["rule_id"] == "CONDUCT_001"),
    None
)

sample_procedure = next(
    (p for p in procedures if p["procedure_id"] == "PROC_008"),
    None
)

sample_deadline = next(
    (d for d in deadlines if d["deadline_id"] == "DEADLINE_001"),
    None
)

# Build evidence packet
test_packet = build_evidence_packet(
    rule_results=[sample_rule] if sample_rule else [],
    procedure_results=[sample_procedure] if sample_procedure else [],
    deadline_results=[sample_deadline] if sample_deadline else []
)

print("Status:", test_packet["status"])
print("Can answer:", test_packet["can_answer"])
print("Evidence count:", test_packet["evidence_count"])

for evidence in test_packet["evidence"]:

    print("\nEvidence ID:", evidence["evidence_id"])
    print("Type:", evidence["evidence_type"])
    print("Title:", evidence["title"])
    print("Information:", evidence["evidence_text"])
    print("Citation:", evidence["citation"])

Status: evidence_found
Can answer: True
Evidence count: 3

Evidence ID: CONDUCT_001
Type: conduct_rule
Title: Academic Dishonesty
Information: Academic dishonesty is defined and dealt with under the Wheel of Academic Law and other applicable University policies and regulations.
Citation: Section 3.1.1, printed page 9

Evidence ID: PROC_008
Type: procedure
Title: Student right to respond
Information: A student against whom an allegation is made must be informed of the nature and details of the allegation and given an opportunity to respond and present relevant evidence before a disciplinary decision is made.
Citation: Section 5.1.3.3.5, printed page 20

Evidence ID: DEADLINE_001
Type: deadline
Title: Submit an appeal against a CDC decision
Information: Submit an appeal against a CDC decision within 10 working days. Trigger: Receipt of the CDC decision.
Citation: Section 5.1.3.3.10, printed page 20


In [14]:
# Test the abstention mechanism
# Simulate a question for which no evidence was retrieved
empty_packet = build_evidence_packet()

print("Status:", empty_packet["status"])
print("Can answer:", empty_packet["can_answer"])
print("System response:", empty_packet["message"])

assert empty_packet["can_answer"] is False

print("\nAbstention test passed.")

Status: insufficient_evidence
Can answer: False
System response: I could not find sufficient supporting information in the available knowledge base. Please contact the appropriate College authority or Student Services Officer.

Abstention test passed.


In [15]:
# Test citation completeness
# Intentionally create an invalid record
invalid_test_record = {
    "procedure_id": "TEST_001",
    "procedure_name": "Test procedure",
    "description": "Example description",
    "section_number": "",
    "source_page": None
}

test_result = validator.validate_record(
    invalid_test_record,
    "procedure"
)

print("Validation result:", test_result["valid"])
print("Issues:", test_result["issues"])

assert test_result["valid"] is False

print("\nCitation and metadata validation test passed.")

Validation result: False
Issues: ['Missing required field: section_number', 'Missing required field: source_page', 'Invalid or missing printed page number.']

Citation and metadata validation test passed.
